In [ ]:
import re
import string
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [ ]:
DATA_PATH = "/content/spam.csv"

def load_data(path):
    try:
        df = pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")

    df = df.dropna(axis=1, how="all")
    df.rename(columns={c: c.strip().lower() for c in df.columns}, inplace=True)

    label_candidates = ["v1", "label", "labels", "category", "class", "target", "type"]
    text_candidates = ["v2", "text", "message", "sms", "body", "content"]
    label_col = next((c for c in label_candidates if c in df.columns), None)
    text_col = next((c for c in text_candidates if c in df.columns), None)

    if label_col is None or text_col is None:
        label_col, text_col = df.columns[0], df.columns[1]

    out = df[[label_col, text_col]].copy()
    out.columns = ["label_raw", "text"]
    out = out.dropna(subset=["text"])
    out["text"] = out["text"].astype(str)
    out["label_raw"] = out["label_raw"].astype(str).str.strip().str.lower()

    label_map = {"ham": 0, "spam": 1, "0": 0, "1": 1, "legitimate": 0, "not spam": 0}
    out["label"] = out["label_raw"].map(label_map)
    out = out.dropna(subset=["label"])
    out["label"] = out["label"].astype(int)
    out = out.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return out[["text", "label"]]

df = load_data(DATA_PATH)
print(f"Loaded {len(df)} messages -> {(df.label==1).sum()} spam / {(df.label==0).sum()} ham")
df.head()

Loaded 5169 messages -> 653 spam / 4516 ham


,text,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
_URL_RE = re.compile(r"http\S+|www\.\S+")
_NUM_RE = re.compile(r"\b\d+\b")
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def clean_text(text):
    text = text.lower()
    text = _URL_RE.sub(" ", text)
    text = _NUM_RE.sub(" number ", text)
    text = text.translate(_PUNCT_TABLE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_pipeline(model_name):
    tfidf = TfidfVectorizer(
        preprocessor=clean_text, stop_words="english",
        ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True,
    )
    if model_name == "nb":
        clf = MultinomialNB()
    elif model_name == "logreg":
        clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=5.0)
    elif model_name == "svm":
        clf = CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0), cv=3)
    else:
        raise ValueError(model_name)
    return Pipeline([("tfidf", tfidf), ("clf", clf)])

MODEL_NAMES = {"nb": "Multinomial Naive Bayes", "logreg": "Logistic Regression", "svm": "Linear SVM"}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

Train: 4135  Test: 1034


In [ ]:
def evaluate(pipeline, name):
    y_pred = pipeline.predict(X_test)
    try:
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    except Exception:
        auc = float("nan")
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": auc,
    }, y_pred

results = []
fitted = {}
for key, name in MODEL_NAMES.items():
    print(f"Training {name} ...")
    pipe = build_pipeline(key)
    pipe.fit(X_train, y_train)
    fitted[key] = pipe
    metrics, y_pred = evaluate(pipe, name)
    results.append(metrics)
    print(f"  Confusion matrix [[TN FP][FN TP]]:\n{confusion_matrix(y_test, y_pred)}\n")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
results_df

Training Multinomial Naive Bayes ...
  Confusion matrix [[TN FP][FN TP]]:
[[902   1]
 [ 27 104]]

Training Logistic Regression ...
  Confusion matrix [[TN FP][FN TP]]:
[[887  16]
 [ 13 118]]

Training Linear SVM ...
  Confusion matrix [[TN FP][FN TP]]:
[[894   9]
 [ 13 118]]



,model,accuracy,precision,recall,f1,roc_auc
2,Linear SVM,0.978723,0.929134,0.900763,0.914729,0.994256
1,Logistic Regression,0.971954,0.880597,0.900763,0.890566,0.992582
0,Multinomial Naive Bayes,0.972921,0.990476,0.793893,0.881356,0.985109


In [ ]:
best_name = results_df.iloc[0]["model"]
best_key = [k for k, v in MODEL_NAMES.items() if v == best_name][0]
best_pipeline = fitted[best_key]

print(f"Best model: {best_name}\n")
print(classification_report(y_test, best_pipeline.predict(X_test), target_names=["ham", "spam"]))

Best model: Linear SVM

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       903
        spam       0.93      0.90      0.91       131

    accuracy                           0.98      1034
   macro avg       0.96      0.95      0.95      1034
weighted avg       0.98      0.98      0.98      1034



In [ ]:
joblib.dump(best_pipeline, "best_sms_spam_model.joblib")
results_df.to_csv("model_comparison.csv", index=False)
print("Saved best_sms_spam_model.joblib and model_comparison.csv")

Saved best_sms_spam_model.joblib and model_comparison.csv


In [ ]:
sample_messages = [
    "Congratulations! You\'ve won a $1000 Walmart gift card. Click here to claim now!",
    "Hey, are you free for dinner tonight?",
    "URGENT: Your account has been suspended, verify now at bit.ly/xyz",
    "Don\'t forget to bring the documents tomorrow",
]

preds = best_pipeline.predict(sample_messages)
probs = best_pipeline.predict_proba(sample_messages)[:, 1]

for msg, pred, prob in zip(sample_messages, preds, probs):
    label = "SPAM" if pred == 1 else "HAM (legit)"
    print(f"[{label}] (spam prob: {prob:.1%})  {msg}")

[SPAM] (spam prob: 96.0%)  Congratulations! You've won a $1000 Walmart gift card. Click here to claim now!
[HAM (legit)] (spam prob: 0.5%)  Hey, are you free for dinner tonight?
[HAM (legit)] (spam prob: 13.7%)  URGENT: Your account has been suspended, verify now at bit.ly/xyz
[HAM (legit)] (spam prob: 2.7%)  Don't forget to bring the documents tomorrow
